- Machine: personal laptop, hostname `liya`
- CPU: 13th Gen Intel Core i5-1340P, 16 logical CPUs (`nproc` = 16)
- OS: Ubuntu, Linux kernel 7.0.0-34-generic, x86_64
- Compiler: gcc 15.2.0
- Compile flags: original Makefile, `-O0` (no optimization), `-pthread -std=c99`

single run: ./src/parallelQuicksort 1000000

Sequential: 0.259 s
Parallel:   0.805 s   <- 3x slower!
Built-in:   0.274 s

Single run can not prove the results yet.

What the code did:
-5 sizes: 100 to 1,000,000
-5 repetitions each
-Runs them in a fixed order, so all the 100s first, then all the 1000s, and so on
-Appends the output to a text file, which a Perl script turns into CSV, then plotted

The weaknesses:  few repetitions,  fixed order, only 5 sizes

run the experiment script: bash scripts/run_benchmarking.sh

results are around below:
size	    Sequential	Parallel
100	        0.00002 s	0.007 s
10,000	    0.0025 s	0.05 s
1,000,000	0.26 s	    0.79 s

Claim is not valid on my laptop.

Test1: 
-10 sizes instead of 5
-30 repetitions instead of 5
-random run order
-writes a CSV directly

In [ ]:
#run_benchmarking2
#!/bin/bash
OUT=data/$(hostname)_$(date +%F)
mkdir -p $OUT
FILE=$OUT/measurements_$(date +%H-%M).csv
echo "size,rep,algo,time" > $FILE

# all (size, repetition) pairs, in random order
for size in 100 300 1000 3000 10000 30000 100000 300000 1000000 3000000; do
  for rep in $(seq 1 30); do
    echo "$size $rep"
  done
done | shuf | while read size rep; do
  ./src/parallelQuicksort $size | awk -v s=$size -v r=$rep '
    /Sequential/ {print s "," r ",sequential," $4}
    /Parallel/   {print s "," r ",parallel," $4}
    /Built-in/   {print s "," r ",builtin," $4}' >> $FILE
done